# 2.2 State Classification

After demodulation, the original notebook uses KMeans to separate the |0> and |1> clusters. In hardware, the online classifier should use fixed centers or thresholds trained offline.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

DATA_PATHS = [
    Path('./s21_data.mat'),
    Path('../s21_data.mat'),
    Path('../../software/s21_data.mat'),
]

def find_s21_data():
    for p in DATA_PATHS:
        if p.exists():
            return p
    raise FileNotFoundError('Put s21_data.mat in the notebook directory or artery/software/.')

def load_s21():
    import scipy.io as sio
    data_path = find_s21_data()
    read_data = sio.loadmat(data_path)
    read_zero = read_data['data'][0]
    read_one = read_data['data'][1]
    read_zero_i, read_zero_q = read_zero[:, :, 0], read_zero[:, :, 1]
    read_one_i, read_one_q = read_one[:, :, 0], read_one[:, :, 1]
    return read_data, read_zero_i, read_zero_q, read_one_i, read_one_q

def demod_part(omega, read_i, read_q, phase=0.0):
    assert read_i.shape == read_q.shape
    ts = np.arange(read_i.shape[1])
    cos_ = np.cos(omega * ts + phase)[None, :]
    sin_ = np.sin(omega * ts + phase)[None, :]
    sum_i = np.sum(read_i * cos_ + read_q * sin_, axis=1)
    sum_q = np.sum(read_q * cos_ - read_i * sin_, axis=1)
    return np.column_stack([sum_i, sum_q])

OMEGAS = 2 * np.pi * (np.array([6.881, 6.79525, 6.97284]) - 7)

In [ ]:
from sklearn.cluster import KMeans
from sklearn import metrics

read_data, read_zero_i, read_zero_q, read_one_i, read_one_q = load_s21()
idx1, idx2 = 1, 2000
result_zero = demod_part(OMEGAS[2], read_zero_i[idx1:idx2], read_zero_q[idx1:idx2])
result_one = demod_part(OMEGAS[2], read_one_i[idx1:idx2], read_one_q[idx1:idx2])
features = np.vstack([result_zero, result_one])
true_labels = np.array([0] * len(result_zero) + [1] * len(result_one))

kmeans = KMeans(n_clusters=2, random_state=0, n_init='auto').fit(features)
labels = kmeans.labels_
acc_direct = metrics.accuracy_score(true_labels, labels)
acc_flip = metrics.accuracy_score(true_labels, 1 - labels)
acc = max(acc_direct, acc_flip)
print('cluster centers:')
print(kmeans.cluster_centers_)
print('best label accuracy:', acc)

plt.figure(figsize=(5, 5))
plt.scatter(features[:, 0], features[:, 1], c=labels, cmap='viridis', s=8, alpha=0.6)
plt.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1], s=120, c='red')
plt.xlabel('integrated I')
plt.ylabel('integrated Q')
plt.tight_layout()

## Original Notebook Figure: KMeans State Clustering

![Original Notebook: KMeans State Clustering](../results/2_2_kmeans_state_clustering.png)

## ARTERY Connection

The trained cluster centers define a low-cost decision surface. The FPGA can compare squared distances to the two centers, or use a projected threshold if the centers are well separated.